# mse-reconstruction-loss — faded example 2: Scale MSE reconstruction loss by a warmup coefficient

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mse-reconstruction-loss`. The last cell reports your progress on the `Generative: MSE reconstruction loss` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: MSE reconstruction loss` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mse-reconstruction-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mse-reconstruction-loss"
DD_SUBTOPIC = "Generative: MSE reconstruction loss"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In VAE training, it is common to ramp the reconstruction loss coefficient from a small value up to 1.0 during a warmup phase. This prevents the model from collapsing to a trivial solution early in training. The standard pattern is to compute the unscaled loss with `F.mse_loss`, then multiply by the current scale factor before adding to other loss terms.

## Faded exercise 2

Implement `scaled_recon_loss(pred, target, scale)` that:
1. Computes the full-batch MSE: `loss = F.mse_loss(pred, target)` (default `reduction='mean'`).
2. Returns `scale * loss`.

Also implement `linear_warmup_scale(step, warmup_steps)` that:
- Returns `min(1.0, step / warmup_steps)` — grows linearly from 0 to 1 over `warmup_steps`.

Your task: **fill in the MSE computation and the scale multiplication in `scaled_recon_loss`**.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch
import torch.nn.functional as F

def linear_warmup_scale(step: int, warmup_steps: int) -> float:
    return min(1.0, step / warmup_steps)

def scaled_recon_loss(pred: torch.Tensor, target: torch.Tensor, scale: float) -> torch.Tensor:
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    pred   = torch.randn(4, 1, 8, 8)
    target = torch.randn(4, 1, 8, 8)
    # scale=1.0 should equal plain MSE
    ref = F.mse_loss(pred, target)
    out = scaled_recon_loss(pred, target, 1.0)
    assert abs(out.item() - ref.item()) < 1e-6
    # scale=0.5 should be half
    out_half = scaled_recon_loss(pred, target, 0.5)
    assert abs(out_half.item() - 0.5 * ref.item()) < 1e-6
    # warmup scale at midpoint
    assert abs(linear_warmup_scale(50, 100) - 0.5) < 1e-9
    assert linear_warmup_scale(200, 100) == 1.0


def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    pred   = torch.randn(4, 1, 8, 8)
    target = torch.randn(4, 1, 8, 8)
    ref = F.mse_loss(pred, target)
    # scale=1.0 matches plain MSE
    assert abs(scaled_recon_loss(pred, target, 1.0).item() - ref.item()) < 1e-6
    # scale=0.0 gives zero
    assert scaled_recon_loss(pred, target, 0.0).item() == 0.0
    # scale=2.0 doubles
    assert abs(scaled_recon_loss(pred, target, 2.0).item() - 2.0 * ref.item()) < 1e-5
    # warmup scale
    assert abs(linear_warmup_scale(25, 100) - 0.25) < 1e-9
    assert linear_warmup_scale(0, 100) == 0.0
    assert linear_warmup_scale(100, 100) == 1.0
    assert linear_warmup_scale(999, 100) == 1.0


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
import torch.nn.functional as F

def linear_warmup_scale(step: int, warmup_steps: int) -> float:
    return min(1.0, step / warmup_steps)

def scaled_recon_loss(pred: torch.Tensor, target: torch.Tensor, scale: float) -> torch.Tensor:
    loss = F.mse_loss(pred, target)
    return scale * loss

def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    pred   = torch.randn(4, 1, 8, 8)
    target = torch.randn(4, 1, 8, 8)
    ref = F.mse_loss(pred, target)
    out = scaled_recon_loss(pred, target, 1.0)
    assert abs(out.item() - ref.item()) < 1e-6
    out_half = scaled_recon_loss(pred, target, 0.5)
    assert abs(out_half.item() - 0.5 * ref.item()) < 1e-6
    assert abs(linear_warmup_scale(50, 100) - 0.5) < 1e-9
    assert linear_warmup_scale(200, 100) == 1.0
```
</details>